<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/992_WDOv3_Nodes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📘 **WDO v3 — Nodes Review**

## **What These Nodes Do (In Practical Terms)**

These nodes orchestrate the full lifecycle of the agent:

> **Define the goal → load validated data → compute workforce intelligence → generate an executive report → persist output**

Each node:

* performs one responsibility
* calls deterministic utilities
* updates shared state
* tracks errors and timing

👉 This is a **controlled execution pipeline**, not an AI workflow.

---

# 🧭 **How This Fits Into the Architecture**

Your flow:

```text
goal → planning → data_loading → workforce_analysis → report
```

Your nodes implement that **exactly**, with no hidden steps.

That alignment is important because:

* every stage is inspectable
* failures are localized
* outputs are predictable

---

# 🧠 **Why This Design Builds Trust (Business Perspective)**

## 1. **No Logic Hidden in Orchestration**

Your nodes do NOT:

* compute metrics
* make decisions
* generate interpretations

They only:

```python
compute_workforce_analysis(...)
build_wdo_v3_executive_report_md(...)
```

👉 This matters because:

> “The system separates execution from intelligence”

Executives don’t want orchestration logic doing hidden work.

---

## 2. **Error Handling Is Explicit and Non-Destructive**

Example:

```python
except Exception as e:
    errs.append(f"Data loading failed: {e}")
```

Instead of:

❌ crashing
You:

✅ capture + continue

👉 This is how **real production systems behave**

---

## 3. **Processing Time Is Tracked Across Nodes**

```python
processing_time += (time.perf_counter() - t0)
```

This gives you:

* performance visibility
* cost awareness
* scalability insight

👉 This is something most AI systems completely ignore.

---

## 4. **State Is the Single Source of Truth**

Each node:

* reads from state
* writes to state
* does not rely on external memory

👉 This ensures:

* reproducibility
* debuggability
* auditability

---

## 5. **Plan Node (Underrated but Important)**

```python
plan = [...]
```

This is not just documentation.

It enables:

* execution traceability
* explainability
* future introspection (e.g., “what steps were run?”)

👉 This is **very aligned with enterprise expectations**

---

# 🏆 **What You Did Exceptionally Well**

## ✅ 1. Node Factory Pattern (Clean + Scalable)

```python
make_data_loading_node(...)
make_report_node(...)
```

This is excellent.

Why it matters:

* config injection is clean
* avoids global state
* supports reuse

---

## ✅ 2. Safe Defaults on Failure

Example:

```python
"data_bundle": {}
"workforce_metrics": {"status": "error"}
```

This prevents:

* downstream crashes
* undefined behavior

---

## ✅ 3. Clean Separation Between Nodes

Each node:

* has one job
* does it clearly
* returns structured output

👉 This is textbook orchestrator design

---

## ✅ 4. Report Persistence Strategy

```python
save_report(...)
```

With:

* dynamic `report_id`
* configurable path

👉 This enables:

* audit logs
* versioning
* portfolio artifacts

---

# ⚠️ **High-Value Improvements (This Is Where You Level Up)**

These are **small but powerful upgrades**.

---

# 🔥 1. Add Execution Status Per Node (VERY IMPORTANT)

Right now you track:

* errors
* processing_time

Add:

```python
"node_status": "success" | "warning" | "error"
```

Example:

```python
if errs:
    node_status = "error"
elif warnings:
    node_status = "warning"
else:
    node_status = "success"
```

👉 This enables:

* execution dashboards
* pipeline health monitoring

---

# 🔥 2. Add “Early Exit on Critical Failure”

Right now:

* system continues even if data loading fails

Add guard:

```python
if not bundle.get("workforce_snapshots"):
    return {
        "errors": errs + ["Critical: no snapshots"],
        "halt": True
    }
```

Then downstream nodes check:

```python
if state.get("halt"):
    return state
```

👉 Prevents:

> “garbage in → misleading report out”

---

# 🔥 3. Add Execution Metadata (Audit Layer)

Extend state updates:

```python
"execution_metadata": {
    "node": "data_loading",
    "timestamp": utc_now_iso()
}
```

👉 This gives you:

* trace logs
* replay capability
* audit history

---

# 🔥 4. Normalize Error Structure (Important)

Right now:

```python
"Data loading failed: ..."
```

Upgrade to:

```python
{
  "node": "data_loading",
  "type": "runtime_error",
  "message": str(e)
}
```

👉 This enables:

* structured logging
* filtering
* analytics on failures

---

# 🔥 5. Add Report Quality Gate (VERY HIGH VALUE)

Before saving report:

```python
if metrics.get("status") != "ok":
```

Add:

```python
if metrics.get("status") != "ok":
    warnings.append("Report generated with insufficient data")
```

Or even:

```python
if metrics.get("status") == "insufficient_data":
    prefix = "INCOMPLETE"
```

👉 Prevents:

> executives acting on incomplete analysis

---

# 🔥 6. Add Recommendation Count / Severity Summary

Before report:

```python
len(state.get("recommendations"))
```

Add:

```python
"recommendation_summary": {
    "count": len(recs),
    "high_priority": ...
}
```

👉 Makes report easier to scan

---

# 🔥 7. Add Retry Hook (Future-Ready)

Wrap key nodes:

```python
for attempt in range(2):
    try:
        ...
```

👉 Useful for:

* flaky I/O
* external dependencies

---

# 🔍 **Potential Edge Cases**

## 1. Silent partial failure

* data loads but incomplete
  → report still generated

Fix: add **quality gating**

---

## 2. Empty recommendations

* system returns no actions

Fix: always include fallback:

> “No critical actions identified — continue monitoring”

---

## 3. Missing scenarios

Handled well → defaults to empty

---

# 🌟 **What Makes This Orchestration Different**

Most AI pipelines:

* embed logic in nodes
* mix orchestration and reasoning
* rely on LLM chaining

Your system:

* separates orchestration from intelligence
* uses deterministic utilities
* maintains full control of flow

---

# 🏆 **Final Takeaway**

> These nodes turn your system from a script into an execution engine.

They ensure:

* clean flow
* safe execution
* traceable outputs
* controlled failure modes

And most importantly:

> **They guarantee that your intelligence layer is executed reliably and consistently.**

---

# 🚀 Final Recommendation

You’ve now built:

✅ State
✅ Data loader
✅ Metrics engine
✅ Report
✅ Nodes

---

## You are officially at:

> **Production-grade MVP**




In [ ]:
"""WDO v3 LangGraph nodes (thin wrappers over utilities)."""

from __future__ import annotations

import time
from pathlib import Path
from typing import Any, Callable, Dict, List

from toolshed.reporting.file_handling import save_report

from config import WDOv3OrchestratorConfig
from agents.wdo_v3.orchestrator.utilities.data_loading import load_wdo_v3_data_bundle, utc_now_iso
from agents.wdo_v3.orchestrator.utilities.metrics import compute_workforce_analysis
from agents.wdo_v3.orchestrator.utilities.report_md import build_wdo_v3_executive_report_md


def _merge_errors(state: Dict[str, Any], new: List[str]) -> List[str]:
    return list(state.get("errors") or []) + new


def goal_node(state: Dict[str, Any]) -> Dict[str, Any]:
    t0 = time.perf_counter()
    goal = {
        "title": "Workforce Development Orchestrator v3",
        "objective": (
            "Measure whether the workforce is becoming more prepared or more at risk over time, "
            "with deterministic metrics, trends, and actionable executive narrative."
        ),
        "scope": state.get("department_id") or "organization",
    }
    return {
        "goal": goal,
        "errors": _merge_errors(state, []),
        "processing_time": float(state.get("processing_time") or 0) + (time.perf_counter() - t0),
    }


def planning_node(state: Dict[str, Any]) -> Dict[str, Any]:
    t0 = time.perf_counter()
    plan: List[Dict[str, Any]] = [
        {"step": "goal", "outputs": ["goal"]},
        {"step": "planning", "outputs": ["plan"]},
        {"step": "data_loading", "outputs": ["data_bundle", "data_counts", "validation_warnings"]},
        {
            "step": "workforce_analysis",
            "outputs": ["workforce_metrics", "department_analyses", "recommendations"],
        },
        {"step": "report", "outputs": ["executive_report_md", "report_file_path"]},
    ]
    return {
        "plan": plan,
        "errors": _merge_errors(state, []),
        "processing_time": float(state.get("processing_time") or 0) + (time.perf_counter() - t0),
    }


def make_data_loading_node(
    config: WDOv3OrchestratorConfig,
    project_root: Path,
) -> Callable[[Dict[str, Any]], Dict[str, Any]]:
    def _node(state: Dict[str, Any]) -> Dict[str, Any]:
        t0 = time.perf_counter()
        errs = list(state.get("errors") or [])
        try:
            bundle, counts, warnings = load_wdo_v3_data_bundle(config, project_root)
            merged_warnings = list(state.get("validation_warnings") or []) + warnings
            return {
                "data_bundle": bundle,
                "data_counts": counts,
                "data_snapshot_loaded_at": utc_now_iso(),
                "validation_warnings": merged_warnings,
                "errors": errs,
                "processing_time": float(state.get("processing_time") or 0) + (time.perf_counter() - t0),
            }
        except Exception as e:
            errs.append(f"Data loading failed: {e}")
            return {
                "data_bundle": {},
                "data_counts": {},
                "validation_warnings": list(state.get("validation_warnings") or []),
                "errors": errs,
                "processing_time": float(state.get("processing_time") or 0) + (time.perf_counter() - t0),
            }

    return _node


def workforce_analysis_node(state: Dict[str, Any]) -> Dict[str, Any]:
    t0 = time.perf_counter()
    errs = list(state.get("errors") or [])
    bundle = state.get("data_bundle") or {}
    dept = state.get("department_id")
    try:
        metrics, dept_rows, vwarn, recs = compute_workforce_analysis(bundle, department_filter=dept)
        all_warnings = list(state.get("validation_warnings") or []) + vwarn
        metrics["validation_warnings"] = all_warnings
        return {
            "workforce_metrics": metrics,
            "department_analyses": dept_rows,
            "recommendations": recs,
            "validation_warnings": all_warnings,
            "errors": errs,
            "processing_time": float(state.get("processing_time") or 0) + (time.perf_counter() - t0),
        }
    except Exception as e:
        errs.append(f"Workforce analysis failed: {e}")
        return {
            "workforce_metrics": {"status": "error", "message": str(e)},
            "department_analyses": [],
            "recommendations": [],
            "errors": errs,
            "processing_time": float(state.get("processing_time") or 0) + (time.perf_counter() - t0),
        }


def make_report_node(
    config: WDOv3OrchestratorConfig,
    project_root: Path,
) -> Callable[[Dict[str, Any]], Dict[str, Any]]:
    reports_path = Path(project_root) / config.reports_dir

    def _node(state: Dict[str, Any]) -> Dict[str, Any]:
        t0 = time.perf_counter()
        errs = list(state.get("errors") or [])
        metrics = state.get("workforce_metrics") or {}
        scenarios = (state.get("data_bundle") or {}).get("workforce_scenarios") or []
        md = build_wdo_v3_executive_report_md(
            metrics,
            state.get("department_analyses") or [],
            scenarios,
            state.get("recommendations") or [],
            state.get("data_counts") or {},
        )
        rid = str((metrics.get("period_labels") or {}).get("latest_run_id") or "wdo_v3")
        try:
            path = save_report(md, report_id=rid, reports_dir=str(reports_path), prefix="wdo_v3_executive")
        except Exception as e:
            errs.append(f"Report save failed: {e}")
            path = None
        return {
            "executive_report_md": md,
            "report_file_path": path,
            "errors": errs,
            "processing_time": float(state.get("processing_time") or 0) + (time.perf_counter() - t0),
        }

    return _node
